In [1]:
import torch
import os
%set_env TOKENIZERS_PARALLELISM=false
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

env: TOKENIZERS_PARALLELISM=false
Using device: cuda


In [2]:
from transformer_lens import HookedESM3,SupportedESM3Config
from esm.pretrained import (
    ESM3_sm_open_v0,
)
from esm.models.esm3 import ESM3
import random
import torch.nn.functional as F
from esm.tokenization import get_esm3_model_tokenizers


In [3]:
from random import randint

def prepare_sequences(proteins_list, device):
    tokenizers = get_esm3_model_tokenizers()
    labels = []  # To store true labels (masked characters)
    masked_indices = []  # To store positions of masked tokens
    masked_proteins = []  # To store masked protein sequences

    # Randomly mask one token per protein
    for protein in proteins_list:
        mask_position = randint(0, len(protein) - 1)
        true_label = protein[mask_position]  # Get the true label (original character)
        labels.append(true_label)
        masked_indices.append(mask_position)

        # Replace the selected position with a mask token
        masked_protein = protein[:mask_position] + tokenizers.sequence.mask_token + protein[mask_position + 1:]
        masked_proteins.append(masked_protein)
    masked_indices = torch.tensor(masked_indices)
    masked_indices+=1 #padding with bos
    # Tokenize the masked sequences
    tokenizers_result = tokenizers.sequence(
        masked_proteins,
        return_tensors="pt",
        add_special_tokens=True,
        padding=True
    )
    input_ids = tokenizers_result['input_ids']
    sequence_ids = tokenizers_result['attention_mask']

    # Tokenize the labels (true tokens)
    tokenized_labels = tokenizers.sequence(
        labels,
        return_tensors="pt",
        add_special_tokens=False,  # No special tokens needed for single characters
        padding=False  # No padding needed for single tokens
    )['input_ids']

    return input_ids.to(device), sequence_ids.to(device), tokenized_labels.squeeze(-1).to(device), masked_indices.to(device)

In [4]:
from esm.tokenization import (
    get_invalid_tokenizer_ids
)
import torch.nn.functional as F
def get_probs_and_log_probs(logits, device, mask_logits_of_invalid_ids=True):
    logits1=logits.clone()
    tokenizer = get_esm3_model_tokenizers().sequence
    mask = torch.ones_like(logits1, dtype=torch.bool, device=device)
    
    if mask_logits_of_invalid_ids:
        valid_ids = list((
                set(tokenizer.all_token_ids)
                - set(tokenizer.special_token_ids)
                - set(get_invalid_tokenizer_ids(tokenizer))
            ))
        mask[:, valid_ids] = False
        logits1[mask] = -torch.inf
    probs = F.softmax(logits1, dim=-1)
    log_probs= logits1.log_softmax(-1)
    return probs, log_probs
    
class Modelresults:
    def __init__(self, sequence_logits, probs,log_probs, probs_mask_invalid, log_probs_mask_invalid, correct_label_probs, correct_label_probs_mask_invalid, correct_label_log_probs, correct_label_log_probs_mask_invalid):
        self.sequence_logits = sequence_logits
        self.probs= probs
        self.log_probs= log_probs
        self.probs_mask_invalid = probs_mask_invalid
        self.log_probs_mask_invalid = log_probs_mask_invalid
        self.correct_label_probs = correct_label_probs
        self.correct_label_probs_mask_invalid = correct_label_probs_mask_invalid
        self.correct_label_log_probs = correct_label_log_probs
        self.correct_label_log_probs_mask_invalid = correct_label_log_probs_mask_invalid

In [5]:
def get_results_on_protein(model, input_ids, sequence_ids, masked_indices, tokenized_labels, device):
    model.eval()
    with torch.no_grad():
        output = model.forward(
        sequence_tokens=input_ids,
        sequence_id=sequence_ids)
        sequence_logits = output.sequence_logits
        batch_indices = torch.arange(sequence_logits.size(0), device=device)
        masked_sequence_logits = sequence_logits[batch_indices, masked_indices]
        probs_mask_invalid, log_probs_mask_invalid = get_probs_and_log_probs(masked_sequence_logits, device, True)
        probs, log_probs= get_probs_and_log_probs(masked_sequence_logits, device, False)
        correct_label_probs = probs[batch_indices, tokenized_labels]
        correct_label_probs_mask_invalid = probs_mask_invalid[batch_indices, tokenized_labels]
        correct_label_log_probs = log_probs[batch_indices, tokenized_labels]
        correct_label_log_probs_mask_invalid = log_probs_mask_invalid[batch_indices, tokenized_labels]
        result =  Modelresults(sequence_logits= masked_sequence_logits,  probs=probs, log_probs=log_probs,probs_mask_invalid=probs_mask_invalid, log_probs_mask_invalid=log_probs_mask_invalid, correct_label_probs=correct_label_probs,
                               correct_label_probs_mask_invalid=correct_label_probs_mask_invalid, correct_label_log_probs=correct_label_log_probs,
                              correct_label_log_probs_mask_invalid=correct_label_log_probs_mask_invalid)
        return result

In [6]:
esm3_original = ESM3_sm_open_v0(device).to(device)
config = SupportedESM3Config(
    use_attn_result=True,
    use_split_qkv_input=True,
    use_hook_mlp_in=False,
    use_attn_in=True,
    esm3_output_type="all",
    esm3_use_torch_layer_norm=True,
    esm3_use_torch_attention_calc=True,
    esm3_use_org_rotary=True
)
esm3_hooked = HookedESM3.from_pretrained(esm_cfg=config, device=device)

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

/home/galkesten/miniconda3/envs/transformer_lens/lib/python3.10/site-packages/esm/pretrained.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(
I

Moving model to device:  cuda
Loaded pretrained model esm3_sm_open_v1 into HookedESM3


In [6]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
file_path ="proteins_faithfulness.csv"
df = pd.read_csv(file_path).head(1000)

class ProteinDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]


protein_dataset = ProteinDataset(df['protein'].tolist())

batch_size = 4
dataloader = DataLoader(protein_dataset, batch_size=batch_size, shuffle=True)



In [7]:
import gc
torch.cuda.empty_cache()
gc.collect()

60

In [25]:
from tqdm import tqdm
import pickle
import torch

original_results_file = "results_original.pkl"
hooked_results_file = "results_hooked.pkl"

# Open files to save results
with open(original_results_file, "wb") as orig_file, open(hooked_results_file, "wb") as hooked_file:
    for batch in tqdm(dataloader, desc="Processing Batches"):
        input_ids, sequence_ids, tokenized_labels, masked_indices = prepare_sequences(batch, device)
        with torch.no_grad():
            # Generate results for both models
            result_original = get_results_on_protein(esm3_original, input_ids, sequence_ids, masked_indices, tokenized_labels, device)
            result_hooked = get_results_on_protein(esm3_hooked, input_ids, sequence_ids, masked_indices, tokenized_labels, device)

            # Save batch results
            pickle.dump(result_original, orig_file)
            pickle.dump(result_hooked, hooked_file)

            # Clean up to save memory
            del result_original
            del result_hooked
            del input_ids, sequence_ids, tokenized_labels, masked_indices
            torch.cuda.empty_cache()  # Clear GPU cache (optional, if using GPU)


Processing Batches: 100%|███████████████████| 250/250 [15:14<00:00,  3.66s/it]


In [40]:

import pickle
# Load results later
original_results_file = "results_original.pkl"
hooked_results_file = "results_hooked.pkl"

with open(original_results_file, "rb") as orig_file, open(hooked_results_file, "rb") as hooked_file:
    results_original = []
    results_hooked = []
    try:
        while True:
            results_original.append(pickle.load(orig_file))
            results_hooked.append(pickle.load(hooked_file))
    except EOFError:
        pass  # End of file


In [41]:
print(len(results_original))
print(len(results_hooked))

250
250


In [42]:
import torch.nn.functional as F
from torch.distributions import Categorical

def top_k_kl_div(logits_p, logits_q, k):
    # Get top-k values and their indices
    top_k_p, indices_p = torch.topk(logits_p, k, dim=-1)
    top_k_q, indices_q = torch.topk(logits_q, k, dim=-1)
    # Assert that both distributions share the same top-k indices
    assert torch.equal(indices_p, indices_q), "Top-k indices of P and Q must match."
    
    log_probs_q = F.log_softmax(top_k_q, dim=-1) 
    probs_p = F.softmax(top_k_p, dim=-1)

    # Compute KL divergence for each batch
    kl_div_batch = F.kl_div(log_probs_q, probs_p, reduction='none').sum(dim=-1)
    return kl_div_batch

In [43]:
import torch
import torch.nn.functional as F

kl_divergences_all = None
kl_divergences_all_top_5 = None
kl_divergences_all_top_10 = None
correct_label_probs_hooked_all = None
correct_label_probs_original_all = None
correct_label_log_probs_hooked_all = None
correct_label_log_probs_original_all = None

max_diff_original = 0.0
max_diff_hooked = 0.0
for result_original, result_hooked in zip(results_original, results_hooked):
    probs_original = result_original.probs
    log_probs_hooked = result_hooked.log_probs
    
    kl_divergence_batch = F.kl_div(log_probs_hooked, probs_original, reduction="none").sum(dim=-1)  # Sum over classes to get divergence per position
    kl_divergences_all = kl_divergence_batch if kl_divergences_all is None else torch.cat((kl_divergences_all, kl_divergence_batch), dim=0)

    kl_divergence_batch_top_5 =top_k_kl_div(result_original.sequence_logits, result_hooked.sequence_logits, 5)
    kl_divergences_all_top_5 = kl_divergence_batch_top_5 if kl_divergences_all_top_5 is None else torch.cat((kl_divergences_all_top_5, kl_divergence_batch_top_5), dim=0)

    kl_divergence_batch_top_10 =top_k_kl_div(result_original.sequence_logits, result_hooked.sequence_logits, 10)
    kl_divergences_all_top_10 = kl_divergence_batch_top_10 if kl_divergences_all_top_10 is None else torch.cat((kl_divergences_all_top_10, kl_divergence_batch_top_10), dim=0)

    correct_label_probs_original = result_original.correct_label_probs
    correct_label_probs_hooked = result_hooked.correct_label_probs
    correct_label_log_probs_original = result_original.correct_label_log_probs
    correct_label_log_probs_hooked = result_hooked.correct_label_log_probs
    
    correct_label_probs_original_all = correct_label_probs_original if correct_label_probs_original_all is None else torch.cat((correct_label_probs_original_all, correct_label_probs_original), dim=0)
    correct_label_probs_hooked_all = correct_label_probs_hooked if correct_label_probs_hooked_all is None else torch.cat((correct_label_probs_hooked_all, correct_label_probs_hooked), dim=0)
    correct_label_log_probs_original_all = correct_label_log_probs_original if correct_label_log_probs_original_all is None else torch.cat((correct_label_log_probs_original_all, correct_label_log_probs_original), dim=0)
    correct_label_log_probs_hooked_all = correct_label_log_probs_hooked if correct_label_log_probs_hooked_all is None else torch.cat((correct_label_log_probs_hooked_all, correct_label_log_probs_hooked), dim=0)

    oringal_diff = torch.max(torch.abs(result_original.probs - result_original.probs_mask_invalid))
    if oringal_diff  > max_diff_original:
        max_diff_original = oringal_diff 
    hooked_diff = torch.max(torch.abs(result_hooked.probs - result_hooked.probs_mask_invalid))
    if hooked_diff > max_diff_hooked:
        max_diff_hooked = hooked_diff
    
print(kl_divergences_all.mean())
print(kl_divergences_all_top_5 .mean())
print(kl_divergences_all_top_10.mean())

faithfulness_1 = correct_label_probs_hooked_all.mean() / correct_label_probs_original_all.mean()
faithfulness_2 = (correct_label_probs_hooked_all / correct_label_probs_original_all).mean()
print(faithfulness_1)
print(faithfulness_2)
faithfulness_3 =  correct_label_log_probs_hooked_all.mean()/ correct_label_log_probs_original_all.mean()
faithfulness_4 = (correct_label_log_probs_hooked_all/ correct_label_log_probs_original_all).mean()
print(faithfulness_3)
print(faithfulness_4)

print(max_diff_original)
print(max_diff_hooked)

print(correct_label_probs_hooked_all.mean())

tensor(2.3136e-09, device='cuda:0')
tensor(4.7128e-09, device='cuda:0')
tensor(3.7411e-09, device='cuda:0')
tensor(1.0000, device='cuda:0')
tensor(1., device='cuda:0')
tensor(1.0000, device='cuda:0')
tensor(1., device='cuda:0')
tensor(5.9605e-08, device='cuda:0')
tensor(2.9802e-08, device='cuda:0')
tensor(0.1688, device='cuda:0')


In [156]:
def prepare_sequences_high_probs(proteins_csv, device):
    tokenizers = get_esm3_model_tokenizers()
    labels = []  # To store true labels (masked characters)
    masked_indices = []  # To store positions of masked tokens
    masked_proteins = []  # To store masked protein sequences

    # Randomly mask one token per protein
    for index, row in proteins_csv.iterrows():
        results_dict = row["results_dict"]
        key = next(iter(results_dict.keys()))
        positions_list = results_dict[key]
        rand = randint(0, len(positions_list) - 1)
        item = positions_list[rand]
        protein = row['seq']
        mask_position = item['masked_position']
        true_label = protein[mask_position]  # Get the true label (original character)
        assert true_label == item['true_label']
        labels.append(true_label)
        masked_indices.append(mask_position)

        # Replace the selected position with a mask token
        masked_protein = protein[:mask_position] + tokenizers.sequence.mask_token + protein[mask_position + 1:]
        masked_proteins.append(masked_protein)
        
    masked_indices = torch.tensor(masked_indices)
    masked_indices+=1 #padding with bos
    # Tokenize the masked sequences
    tokenizers_result = tokenizers.sequence(
        masked_proteins,
        return_tensors="pt",
        add_special_tokens=True,
        padding=True
    )
    input_ids = tokenizers_result['input_ids']
    sequence_ids = tokenizers_result['attention_mask']

    # Tokenize the labels (true tokens)
    tokenized_labels = tokenizers.sequence(
        labels,
        return_tensors="pt",
        add_special_tokens=False,  # No special tokens needed for single characters
        padding=False  # No padding needed for single tokens
    )['input_ids']

    return input_ids.to(device), sequence_ids.to(device), tokenized_labels.squeeze(-1).to(device), masked_indices.to(device)

In [157]:
import pandas as pd
import json
from torch.utils.data import Dataset, DataLoader
file_path ="proteins_faithfulness_high_prob.csv"
df = pd.read_csv(file_path).head(1000)
df["results_dict"] = df["results_dict"].apply(json.loads)

first = df.iloc[0]["results_dict"] 
key = next(iter(first.keys()))
lis1= first[key]
rand= randint(0, len(lis1) - 1)
mask_position =lis1[rand]
mask_position

{'masked_position': 171,
 'true_label': 'K',
 'predicted_label': 'K',
 'correct_label_probability': 0.893265426158905,
 'predicted_label_probability': 0.893265426158905,
 'is_correct': True}

In [158]:
from tqdm import tqdm
import pickle
import torch

# File paths for saving results
original_results_file_high_prob = "results_original_high_prob.pkl"
hooked_results_file_high_prob = "results_hooked_high_prob.pkl"

# Define batch size
batch_size = 4

# Open files to save results
with open(original_results_file_high_prob, "wb") as orig_file, open(hooked_results_file_high_prob, "wb") as hooked_file:
    for i in tqdm(range(0, len(df), batch_size), desc="Processing Batches"):
        # Slice DataFrame to get a batch of size 4
        batch = df.iloc[i:i + batch_size]

        # Prepare input sequences
        input_ids, sequence_ids, tokenized_labels, masked_indices = prepare_sequences_high_probs(batch, device)

        with torch.no_grad():
            # Generate results for both models
            result_original = get_results_on_protein(
                esm3_original, input_ids, sequence_ids, masked_indices, tokenized_labels, device
            )
            result_hooked = get_results_on_protein(
                esm3_hooked, input_ids, sequence_ids, masked_indices, tokenized_labels, device
            )

            # Save batch results
            pickle.dump(result_original, orig_file)
            pickle.dump(result_hooked, hooked_file)

            # Clean up to save memory
            del result_original
            del result_hooked
            del input_ids, sequence_ids, tokenized_labels, masked_indices
            torch.cuda.empty_cache()  # Clear GPU cache (optional, if using GPU)


Processing Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 221/221 [13:53<00:00,  3.77s/it]


In [44]:

import pickle
# Load results later
original_results_file_high_prob = "results_original_high_prob.pkl"
hooked_results_file_high_prob = "results_hooked_high_prob.pkl"

with open(original_results_file_high_prob , "rb") as orig_file, open(hooked_results_file_high_prob, "rb") as hooked_file:
    results_original = []
    results_hooked = []
    try:
        while True:
            results_original.append(pickle.load(orig_file))
            results_hooked.append(pickle.load(hooked_file))
    except EOFError:
        pass  # End of file


In [45]:
len(results_original)
len(results_hooked )

221

In [46]:
import torch
import torch.nn.functional as F

kl_divergences_all = None
kl_divergences_all_top_5 = None
kl_divergences_all_top_10 = None
correct_label_probs_hooked_all = None
correct_label_probs_original_all = None
correct_label_log_probs_hooked_all = None
correct_label_log_probs_original_all = None

max_diff_original = 0.0
max_diff_hooked = 0.0
for result_original, result_hooked in zip(results_original, results_hooked):
    log_probs_original = result_original.log_probs
    log_probs_hooked = result_hooked.log_probs
    
    kl_divergence_batch = F.kl_div(log_probs_hooked, log_probs_original, reduction="none",log_target=True).sum(dim=-1)  # Sum over classes to get divergence per position
    kl_divergences_all = kl_divergence_batch if kl_divergences_all is None else torch.cat((kl_divergences_all, kl_divergence_batch), dim=0)

    kl_divergence_batch_top_5 =top_k_kl_div(result_original.sequence_logits, result_hooked.sequence_logits, 5)
    kl_divergences_all_top_5 = kl_divergence_batch_top_5 if kl_divergences_all_top_5 is None else torch.cat((kl_divergences_all_top_5, kl_divergence_batch_top_5), dim=0)

    kl_divergence_batch_top_10 =top_k_kl_div(result_original.sequence_logits, result_hooked.sequence_logits, 10)
    kl_divergences_all_top_10 = kl_divergence_batch_top_10 if kl_divergences_all_top_10 is None else torch.cat((kl_divergences_all_top_10, kl_divergence_batch_top_10), dim=0)

    correct_label_probs_original = result_original.correct_label_probs
    correct_label_probs_hooked = result_hooked.correct_label_probs
    correct_label_log_probs_original = result_original.correct_label_log_probs
    correct_label_log_probs_hooked = result_hooked.correct_label_log_probs
    
    correct_label_probs_original_all = correct_label_probs_original if correct_label_probs_original_all is None else torch.cat((correct_label_probs_original_all, correct_label_probs_original), dim=0)
    correct_label_probs_hooked_all = correct_label_probs_hooked if correct_label_probs_hooked_all is None else torch.cat((correct_label_probs_hooked_all, correct_label_probs_hooked), dim=0)
    correct_label_log_probs_original_all = correct_label_log_probs_original if correct_label_log_probs_original_all is None else torch.cat((correct_label_log_probs_original_all, correct_label_log_probs_original), dim=0)
    correct_label_log_probs_hooked_all = correct_label_log_probs_hooked if correct_label_log_probs_hooked_all is None else torch.cat((correct_label_log_probs_hooked_all, correct_label_log_probs_hooked), dim=0)

    oringal_diff = torch.max(torch.abs(result_original.probs - result_original.probs_mask_invalid))
    if oringal_diff  > max_diff_original:
        max_diff_original = oringal_diff 
    hooked_diff = torch.max(torch.abs(result_hooked.probs - result_hooked.probs_mask_invalid))
    if hooked_diff > max_diff_hooked:
        max_diff_hooked = hooked_diff
    
print(kl_divergences_all.mean())
print(kl_divergences_all_top_5 .mean())
print(kl_divergences_all_top_10.mean())

faithfulness_1 = correct_label_probs_hooked_all.mean() / correct_label_probs_original_all.mean()
faithfulness_2 = (correct_label_probs_hooked_all / correct_label_probs_original_all).mean()
print(faithfulness_1)
print(faithfulness_2)
faithfulness_3 =  correct_label_log_probs_hooked_all.mean()/ correct_label_log_probs_original_all.mean()
faithfulness_4 = (correct_label_log_probs_hooked_all/ correct_label_log_probs_original_all).mean()
print(faithfulness_3)
print(faithfulness_4)

print(max_diff_original)
print(max_diff_hooked)

print(correct_label_probs_hooked_all.mean())

tensor(1.7898e-09, device='cuda:0')
tensor(-3.6548e-09, device='cuda:0')
tensor(2.1333e-09, device='cuda:0')
tensor(1.0000, device='cuda:0')
tensor(1., device='cuda:0')
tensor(1., device='cuda:0')
tensor(1.0000, device='cuda:0')
tensor(2.0515e-10, device='cuda:0')
tensor(2.0515e-10, device='cuda:0')
tensor(0.7572, device='cuda:0')
